# **READ ME FILE**

This code performs Linear Regression and Naive Bayes classification on the Telco Customer Churn dataset for classification and compares the two algorithms performance on Accuracy, Precision, Recall, F1-score and ROC-AUC.

Required libraries:
* numpy as np
* pandas as pd
* matplotlib.pyplot as plt
* sklearn
* sklearn.preprocessing - StandardScaler
* cv2
* sklearn.naive_bayes - GaussianNB
* sklearn - metrics
* sklearn.model_selection import cross_validate

**IMPORTANT NOTE**
To run this code in your environment you must allow Google Colab to mount to your drive. Before running the code you must change your base directory to the location of the data files in the section labelled "CHANGE BASE DIRECTORY HERE TO YOUR GDRIVE FILE LOCATION" in the first code block.



**Mounting Google Drive + Importing Libraries**

In [ ]:
## mounting Google Drive for file import ##
####### you must allow google to mount to your drive ##########
from google.colab import drive
drive.mount('/content/drive')

######## CHANGE TO BASE DIRECTORY HERE TO YOUR GDRIVE FILE LOCATION ########
%cd /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/


In [ ]:
## importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
import cv2
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
from sklearn.model_selection import cross_validate

Importing the dataset:

**Declaration of gen AI usage!** I could see missing values in the "TotalCharges" column in excel, but pandas was not catching them below in my .isna().sum(). I added in the na_values = ... from the help of chat gpt!

In [ ]:
customer_df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv',
                          na_values=[" ", "", "NA", "NaN", "?"]) ## ai usage here
print(f"# rows: {customer_df.shape[0]}, # cols: {customer_df.shape[1]}")
print("\nRetrieving column names: ")
print(customer_df.columns)
print("\nThere are 7,043 samples, 20 feature variables, and target = 'Churn'")


# rows: 7043, # cols: 21

Retrieving column names: 
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

There are 7,043 samples, 20 feature variables, and target = 'Churn'


**Data Cleaning**

In [ ]:
print(customer_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Cleaning the data to be all numerical for future logistic regression.

Current Scheme + handling process:


*   customerID: drop completely (not needed)
*   gender: Male/Female; map to binary
*   SeniorCitizen: 0/1; leave as is
*   Partner: No/Yes; map to binary
*   Dependents: No/Yes; map to binary
*   tenure: Int; leave as is
*   PhoneService: No/Yes; map to binary
*   MultipleLines: No phone service/No/Yes; map to tertiary
*   InternetService: DSL/fiber optic/No; map to tertiary
*   OnlineSecurity: No/Yes/No internet service; map to tertiary
*   OnlineBackup: No/Yes/No internet service; map to tertiary
*   DeviceProtection: No/Yes/No internet service; map to tertiary
*   TechSupport: No/Yes/No internet service; map to tertiary
*   StreamingTV: No/Yes/No internet service; map to tertiary
*   StreamingMovies: No/Yes/No internet service; map to tertiary
*   Contract: Month-to-month, One year, Two year; map to tertiary
*   PaperlessBilling: No/Yes; map to binary
*   PaymentMethod: Bank transfer (automatic), Electronic check, Mailed check, Credit card (automatic); map to 0/1/2/3
*   MonthlyCharges: float64; leave as is
*   TotalCharges: float64; leave as is
*   Churn: No/Yes; map to binary



In [ ]:
## mapping categorical to integers
customer_df = customer_df.drop('customerID', axis =1)
customer_df['gender'] = customer_df['gender'].map({'Male': 0, 'Female': 1})
customer_df['Partner'] = customer_df['Partner'].map({'No': 0, 'Yes': 1})
customer_df['Dependents'] = customer_df['Dependents'].map({'No': 0, 'Yes': 1})
customer_df['PhoneService'] = customer_df['PhoneService'].map({'No': 0, 'Yes': 1})
customer_df['MultipleLines'] = customer_df['MultipleLines'].map({'No': 0, 'Yes':1,'No phone service': 2})
customer_df['InternetService'] = customer_df['InternetService'].map({'No': 0, 'DSL':1,'Fiber optic': 2})
customer_df['OnlineSecurity'] = customer_df['OnlineSecurity'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['OnlineBackup'] = customer_df['OnlineBackup'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['DeviceProtection'] = customer_df['DeviceProtection'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['TechSupport'] = customer_df['TechSupport'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['StreamingTV'] = customer_df['StreamingTV'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['StreamingMovies'] = customer_df['StreamingMovies'].map({'No': 0, 'Yes':1,'No internet service': 2})
customer_df['Contract'] = customer_df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year':2})
customer_df['PaperlessBilling'] = customer_df['PaperlessBilling'].map({'No': 0, 'Yes': 1})
customer_df['PaymentMethod'] = customer_df['PaymentMethod'].map({'Credit card (automatic)': 0,
                                                                 'Bank transfer (automatic)': 1,
                                                                 'Electronic check': 2,
                                                                'Mailed check': 3})
customer_df['Churn'] = customer_df['Churn'].map({'No': 0, 'Yes': 1})

print(customer_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   int64  
 7   InternetService   7043 non-null   int64  
 8   OnlineSecurity    7043 non-null   int64  
 9   OnlineBackup      7043 non-null   int64  
 10  DeviceProtection  7043 non-null   int64  
 11  TechSupport       7043 non-null   int64  
 12  StreamingTV       7043 non-null   int64  
 13  StreamingMovies   7043 non-null   int64  
 14  Contract          7043 non-null   int64  
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   int64  


In [ ]:
## checking for missing values
print(f"Number of missing attributes: {customer_df.isna().sum()}")

### Imputing with two methods: ###

# method 1: zero impuatation
customer_df_zero = customer_df.fillna(0)

# method 2: dropping Total Charges feature
customer_df_drop = customer_df.drop('TotalCharges', axis=1)

### scaling the numerical features to mean 0, sd 1 ###
scaler = StandardScaler()

# tenure
customer_df_zero['tenure'] = scaler.fit_transform(customer_df_zero[['tenure']])
customer_df_drop['tenure'] = scaler.fit_transform(customer_df_drop[['tenure']])

# MonthlyCharges
customer_df_zero['MonthlyCharges'] = scaler.fit_transform(customer_df_zero[['MonthlyCharges']])
customer_df_drop['MonthlyCharges'] = scaler.fit_transform(customer_df_drop[['MonthlyCharges']])

# TotalCharges - zero imp only
customer_df_zero['TotalCharges'] = scaler.fit_transform(customer_df_zero[['TotalCharges']])


Number of missing attributes: gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


Splitting the data into feature and target dataframes.

In [ ]:
## zero imputation set
zero_imp_x = customer_df_zero.drop('Churn', axis = 1)
zero_imp_y = customer_df_zero['Churn']

## drop imputation set
drop_imp_x = customer_df_drop.drop('Churn', axis = 1)
drop_imp_y = customer_df_drop['Churn']


**Part 1: Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import cross_validate

## defining evaluation metrics ##
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

############## Method 1: Zero Imputation ################

## training and making predictions with the model
lm_zero = LogisticRegression(max_iter=10000, penalty='l1', solver='liblinear', C=0.7) #manually changed these
lm_zero.fit(zero_imp_x, zero_imp_y)

y_pred_zero = lm_zero.predict(zero_imp_x)


## applying cross validation
cv_zero = cross_validate(lm_zero, zero_imp_x, zero_imp_y, cv=5, scoring=scoring)

cv_acc_zero = cv_zero['test_accuracy'].mean()
cv_prec_zero = cv_zero['test_precision'].mean()
cv_rec_zero = cv_zero['test_recall'].mean()
cv_f1_zero = cv_zero['test_f1'].mean()
cv_roc_zero = cv_zero['test_roc_auc'].mean()


print("Logistic Regression, Method 1 (zero imp) Results")
print("-------------------------------------------------")
print(f"Accuracy = {cv_acc_zero}")
print(f"Precision = {cv_prec_zero}")
print(f"Recall = {cv_rec_zero}")
print(f"F1 score = {cv_f1_zero}")
print(f"ROC-AUC = {cv_roc_zero}")


############## Method 2: Dropping Feature ################

## training and making predictions with the model
lm_drop = LogisticRegression(max_iter=10000, penalty='l1', solver='liblinear', C=0.7) # matched method 1 for consistency
lm_drop.fit(drop_imp_x, drop_imp_y)

y_pred_drop = lm_drop.predict(drop_imp_x)


## applying cross validation
cv_drop = cross_validate(lm_drop, drop_imp_x, drop_imp_y, cv=5, scoring=scoring)

cv_acc_drop = cv_drop['test_accuracy'].mean()
cv_prec_drop = cv_drop['test_precision'].mean()
cv_rec_drop = cv_drop['test_recall'].mean()
cv_f1_drop = cv_drop['test_f1'].mean()
cv_roc_drop = cv_drop['test_roc_auc'].mean()


print("\nLogistic Regression, Method 2 (drop feature) Results")
print("-------------------------------------------------")
print(f"Accuracy = {cv_acc_drop}")
print(f"Precision = {cv_prec_drop}")
print(f"Recall = {cv_rec_drop}")
print(f"F1 score = {cv_f1_drop}")
print(f"ROC-AUC = {cv_roc_drop}")



Logistic Regression, Method 1 (zero imp) Results
-------------------------------------------------
Accuracy = 0.8047713763791213
Precision = 0.6578109001814713
Recall = 0.5505641496179265
F1 score = 0.5993847756735036
ROC-AUC = 0.8445845570992514

Logistic Regression, Method 2 (drop feature) Results
-------------------------------------------------
Accuracy = 0.8047712755661657
Precision = 0.6589361842066422
Recall = 0.5473570271393959
F1 score = 0.5979131279507002
ROC-AUC = 0.8425067613403598


**Naive Bayes**


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
from sklearn.model_selection import cross_validate

## defining evaluation metrics ##
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

############## Method 1: Zero Imputation ################

## training and making predictions with the model
nb_zero = GaussianNB()
nb_zero.fit(zero_imp_x, zero_imp_y)

nb_pred_zero = nb_zero.predict(zero_imp_x)


## applying cross validation
cv_zero = cross_validate(nb_zero, zero_imp_x, zero_imp_y, cv=5, scoring=scoring)

cv_acc_zero = cv_zero['test_accuracy'].mean()
cv_prec_zero = cv_zero['test_precision'].mean()
cv_rec_zero = cv_zero['test_recall'].mean()
cv_f1_zero = cv_zero['test_f1'].mean()
cv_roc_zero = cv_zero['test_roc_auc'].mean()


print("Naive Bayes Classifier, Method 1 (zero imp) Results")
print("-------------------------------------------------")
print(f"Accuracy = {cv_acc_zero}")
print(f"Precision = {cv_prec_zero}")
print(f"Recall = {cv_rec_zero}")
print(f"F1 score = {cv_f1_zero}")
print(f"ROC-AUC = {cv_roc_zero}")


############## Method 2: Dropping Feature ################

## training and making predictions with the model
nb_drop = GaussianNB()
nb_drop.fit(drop_imp_x, drop_imp_y)

nb_pred_drop = nb_drop.predict(drop_imp_x)


## applying cross validation
cv_drop = cross_validate(nb_drop, drop_imp_x, drop_imp_y, cv=5, scoring=scoring)

cv_acc_drop = cv_drop['test_accuracy'].mean()
cv_prec_drop = cv_drop['test_precision'].mean()
cv_rec_drop = cv_drop['test_recall'].mean()
cv_f1_drop = cv_drop['test_f1'].mean()
cv_roc_drop = cv_drop['test_roc_auc'].mean()


print("\nNaive Bayes Classifier, Method 2 (drop feature) Results")
print("-------------------------------------------------")
print(f"Accuracy = {cv_acc_drop}")
print(f"Precision = {cv_prec_drop}")
print(f"Recall = {cv_rec_drop}")
print(f"F1 score = {cv_f1_drop}")
print(f"ROC-AUC = {cv_roc_drop}")




Naive Bayes Classifier, Method 1 (zero imp) Results
-------------------------------------------------
Accuracy = 0.7082205908445707
Precision = 0.47146848677293257
Recall = 0.8218190420209028
F1 score = 0.5991442677998469
ROC-AUC = 0.8144310102222784

Naive Bayes Classifier, Method 2 (drop feature) Results
-------------------------------------------------
Accuracy = 0.7016890202593716
Precision = 0.46517196956877216
Recall = 0.8293085403793494
F1 score = 0.5959919865310436
ROC-AUC = 0.8141136074629085


**Discussion:**

For my implementation, I started by cleaning the dataset by encoding the classification based parameters into numeric values since Logistic Regression and Naive Bayes classification cannot ingest strings as inputs. Next, I found that there were 11 missing values in the "TotalCharges" feature column. I handled this with two methods: Method 1 - zero imputation, and Method 2 - dropping the feature completely. I wanted to try a run with the drop feature method specifically to see how much impact it was contributing to the model. I decided to standardize the numerical features (tenure, MonthlyCharges, and TotalCharges) to make sure that the model wouldn't be impacted by the features being on different scales.

Next, I fit the data with Logistic Regression and Naive Bayes classification. For Logistic Regression, I tried no regularization, L1 regularization, and L2 regularization with varying C values and found that my highest accuracy occured with L1 regularization, C = 0.7, and the liblinear solver. I used the same hyperparameters for both imputation methods for consistency for later comparison. 10,000 iterations were used for both applications. Comparing my two imputation methods, I didnt't see a significant difference in any of my accuracy metrics (accuracy, precision, recall, F1-score, or ROC-AUC). This shows that the TotalCharges feature is not impactful to the model, which might be due to covariance between other features or that the other features are sufficient to cover the trends of the data.

Using the GaussianNB library from scikit learn, I fit both datasets for a Naive Bayes classification. Similar to before, there were minor differences between the results for both imputation methods, showing again that the TotalCharges feature did not have significant impact on the performance of the model. I did not tune with hyperparameters since Naive Bayes is a calculate probabilities and compute method, and doesn't need tuning on the training phase.

Comparing the two algorithms, I found that Logistic Regression performed significantly better than Naive Bayes classification on Accuracy and Precision. However, both models showed similar performance in F1-score and ROC-AUC and Logistic Regression performed much worse on Recall.

Logistic Regression vs. Naive Bayes:
*   Accuracy: +10 ppts
*   Precision: +18 ppts
*   Recall: -27 ppts
*   F1-score: +/- 0 ppts
*   ROC-AUC: +3 ppts

These results happen because Logistic Regression can be impacted by imbalanced datasets, especially when the positive target (Churn) is less common than the negative target (No Churn). In this dataset, there are 5174 negative targets and 1869 positive targets, which helps explain why Logistic Regression performed worse on recall. Another positive of the Naive Bayes classifier is that it doesn't have a deep training phase, so it is computationally light compared to Logistic Regression. However, Naive Bayes assumes that all of the samples are independent, which might not be the case in the nature of this dataset. This shows in the Accuracy and Precision performance of the model.


